Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from tqdm.auto import tqdm

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Load project modules

In [ ]:
from src.data_loader import load_all_train_data
from src.normalization import normalize_dataframe
from src.blocking import (
    add_blocking_keys,
    generate_all_candidates,
)

print("✓ Modules loaded")

Loading training data...

========== LOADING TRAIN DATA ==========


Train files:   0%|          | 0/4 [00:00<?, ?file/s]


Loading: train_source1.tsv


Train files:  25%|██▌       | 1/4 [00:06<00:18,  6.19s/file]

  ✓ Loaded 2,206,821 rows × 4 columns

Loading: train_source2.tsv


...

  ✓ Loaded 5,034,616 rows × 4 columns

Loading: train_source3.tsv


Train files:  75%|███████▌  | 3/4 [00:39<00:14, 14.42s/file]

  ✓ Loaded 5,285,603 rows × 4 columns

Loading: train_ground_truth.tsv


Train files: 100%|██████████| 4/4 [00:43<00:00, 10.96s/file]

  ✓ Loaded 2,206,821 rows × 2 columns

✓ Training data loaded in 53.43 seconds
source1        : 2,206,821 rows × 4 columns
source2        : 5,034,616 rows × 4 columns
source3        : 5,285,603 rows × 4 columns
ground_truth   : 2,206,821 rows × 2 columns


Load training data

In [ ]:
print("Loading training data...")

train_data = load_all_train_data()

source1_all = train_data["source1"]
source2 = train_data["source2"]
source3 = train_data["source3"]
ground_truth = train_data["ground_truth"]

print("\n========== DATASET SIZES ==========")

print(f"Source1       : {len(source1_all):,}")
print(f"Source2       : {len(source2):,}")
print(f"Source3       : {len(source3):,}")
print(f"Ground truth  : {len(ground_truth):,}")

Ground truth shape: (2206821, 2)

Columns:
['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


Inspect ground truth

In [ ]:
print("========== GROUND TRUTH ==========")

print(ground_truth.head())

print("\nColumns:")
print(ground_truth.columns.tolist())

print("\nExample matched IDs:")
print(ground_truth["matched_entity_ids"].head(10).to_string(index=False))

Parsing ground truth:   0%|          | 0/2206821 [00:00<?, ?it/s]

Parsing ground truth: 100%|██████████| 2206821/2206821 [01:36<00:00, 22810.80it/s]



✓ Ground-truth pairs: 7,638,365


,source1_entity_id,matched_entity_id
0,S1-965667,S2-681193310
1,S1-965667,S2-743505751
2,S1-965667,S3-775321672
3,S1-965667,S3-11291185
4,S1-965667,S3-860443364
5,S1-55344266,S2-249013014
6,S1-55344266,S2-197070651
7,S1-55344266,S3-478195123
8,S1-55344266,S3-384364074
9,S1-343815751,S2-790675320


Parse ground truth

In [ ]:
def parse_ground_truth(df):
    rows = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Parsing ground truth"
    ):
        source1_id = row["source1_entity_id"]
        matched_ids = row["matched_entity_ids"]

        if not isinstance(matched_ids, str):
            continue

        for target_id in matched_ids.split(","):
            target_id = target_id.strip()

            if target_id:
                rows.append({
                    "source1_entity_id": source1_id,
                    "matched_entity_id": target_id,
                })

    return pd.DataFrame(rows)


ground_truth_pairs = parse_ground_truth(ground_truth)

print("\nTotal ground-truth pairs:", f"{len(ground_truth_pairs):,}")
print(ground_truth_pairs.head())

target_source
S3    3944746
S2    3693619
Name: count, dtype: int64


Identify target source

In [ ]:
ground_truth_pairs["target_source"] = (
    ground_truth_pairs["matched_entity_id"]
    .str.extract(r"^(S[23])-", expand=False)
)

print(
    ground_truth_pairs["target_source"]
    .value_counts()
)

Ground-truth Source1 IDs selected: 100,000
Source1 records found: 100,000


Check ground-truth pair counts

In [ ]:
print("========== GROUND TRUTH PAIRS ==========")

print(
    ground_truth_pairs["target_source"]
    .value_counts()
    .sort_index()
)

print("\nS2 pairs:")
print(
    f"{(ground_truth_pairs['target_source'] == 'S2').sum():,}"
)

print("\nS3 pairs:")
print(
    f"{(ground_truth_pairs['target_source'] == 'S3').sum():,}"
)

Ground-truth pairs for sample: 366,464


,source1_entity_id,matched_entity_id,target_source
0,S1-965667,S2-681193310,S2
1,S1-965667,S2-743505751,S2
2,S1-965667,S3-775321672,S3
3,S1-965667,S3-11291185,S3
4,S1-965667,S3-860443364,S3
5,S1-55344266,S2-249013014,S2
6,S1-55344266,S2-197070651,S2
7,S1-55344266,S3-478195123,S3
8,S1-55344266,S3-384364074,S3
9,S1-343815751,S2-790675320,S2


Create validation Source1 sample

In [ ]:
SAMPLE_SIZE = 100_000

ground_truth_source1_ids = (
    ground_truth_pairs["source1_entity_id"]
    .drop_duplicates()
)

sample_source1_ids = (
    ground_truth_source1_ids
    .head(SAMPLE_SIZE)
)

source1 = source1_all[
    source1_all["entity_id"].isin(sample_source1_ids)
].copy()

print("========== VALIDATION SAMPLE ==========")

print(
    f"Requested Source1 sample : {SAMPLE_SIZE:,}"
)

print(
    f"Actual Source1 sample    : {len(source1):,}"
)

Source2: 5,034,616 rows
Source3: 5,285,603 rows


Restrict ground truth to validation sample

In [ ]:
sample_truth = ground_truth_pairs[
    ground_truth_pairs["source1_entity_id"].isin(
        sample_source1_ids
    )
].copy()

print("========== SAMPLE GROUND TRUTH ==========")

print(
    f"Total sample pairs : {len(sample_truth):,}"
)

print(
    sample_truth["target_source"]
    .value_counts()
    .sort_index()
)

Normalizing Source1...
✓ Source1 normalized in 5.16 seconds


Verify Source1 sample

In [ ]:
print("========== SOURCE1 SAMPLE ==========")

print(source1.head())

print("\nShape:")
print(source1.shape)

print("\nColumns:")
print(source1.columns.tolist())

Normalizing Source2...
✓ Source2 normalized in 129.08 seconds


Normalize Source1

In [ ]:
print("Normalizing Source1...")

source1 = normalize_dataframe(source1)

print("✓ Source1 normalized")

Normalizing Source3...
✓ Source3 normalized in 132.32 seconds


Normalize Source2

In [ ]:
print("Normalizing Source2...")

source2 = normalize_dataframe(source2)

print("✓ Source2 normalized")

Creating Source1 blocking keys...
Creating Source2 blocking keys...
Creating Source3 blocking keys...

✓ Blocking keys created


Normalize Source3

In [ ]:
print("Normalizing Source3...")

source3 = normalize_dataframe(source3)

print("✓ Source3 normalized")

,entity_id,business_name,business_name_normalized,business_address_normalized,country_normalized,name_block_key,address_block_key
12,S1-236312382,"Kelly Advisory, Inc",kelly advisory inc,301 1st street chokio mn,us,kelly_inc,301_1st_street
41,S1-993527714,George Saul Inc,george saul inc,unit unit 367 1400 great wolf drive wi village...,us,george_inc,unit_unit_367
142,S1-269039848,Holy Community Church,holy community church,5546 cincinnati dayton road middletown oh,us,holy_church,5546_cincinnati_dayton
190,S1-136894582,Economic Development United Fellowship,economic development united fellowship,3841 forsyth park court greenville nc,us,economic_fellowship,3841_forsyth_park
227,S1-259452306,Shorter and Reyes Xrp,shorter and reyes xrp,130 main street durham nc,us,shorter_xrp,130_main_street
278,S1-638466806,Radhey Basket Private Limited,radhey basket private limited,panchratna office no 624 1 karve road erandawa...,india,radhey_limited,panchratna_office_no
295,S1-436985273,Corner Pilates,corner pilates,ky 116 redwood drive frankfort,us,corner_pilates,ky_116_redwood
298,S1-169008536,Urban Projects,urban projects,c o alauddin molla noorpur ramnagar 24 pargana...,india,urban_projects,c_o_alauddin
339,S1-522840559,Tulsa Pediatric Dentistry Continental Medicine,tulsa pediatric dentistry continental medicine,7085 foxtail lane tulsa ok,us,tulsa_medicine,7085_foxtail_lane
447,S1-447452795,Blue Network Corporation,blue network corporation,177 1st avenue nashville tn,us,blue_corporation,177_1st_avenue


Check normalized columns

In [ ]:
print("========== NORMALIZED COLUMNS ==========")

print("Source1:")
print(source1.columns.tolist())

print("\nSource2:")
print(source2.columns.tolist())

print("\nSource3:")
print(source3.columns.tolist())

Generating Source1 → Source2 name candidates...
✓ Completed in 39.72 seconds
Candidates: 28,367,536


Generate blocking keys for Source1

In [ ]:
print("Generating Source1 blocking keys...")

source1 = add_blocking_keys(source1)

print("✓ Source1 blocking keys generated")

Generating Source1 → Source2 address candidates...
✓ Completed in 12.27 seconds
Candidates: 3,436,771


Generate blocking keys for Source2

In [ ]:
print("Generating Source2 blocking keys...")

source2 = add_blocking_keys(source2)

print("✓ Source2 blocking keys generated")

Source1 → Source2 candidates: 31,784,568


Generate blocking keys for Source3

In [ ]:
print("Generating Source3 blocking keys...")

source3 = add_blocking_keys(source3)

print("✓ Source3 blocking keys generated")

Generating Source1 → Source3 name candidates...
✓ Completed in 34.91 seconds
Candidates: 35,645,473


Inspect blocking keys

In [ ]:
blocking_columns = [
    "country_block_key",
    "name_block_key",
    "name_first_key",
    "name_last_key",
    "name_length_key",
    "address_block_key",
    "address_first_key",
    "address_last_key",
]

print("========== BLOCKING KEY SAMPLE ==========")

print(
    source1[
        [
            "entity_id",
            "business_name",
            "business_name_normalized",
            "country_normalized",
            *blocking_columns,
        ]
    ].head(20).to_string(index=False)
)

Generating Source1 → Source3 address candidates...
✓ Completed in 13.55 seconds
Candidates: 3,309,844


Check blocking-key statistics

In [ ]:
print("========== BLOCKING KEY STATISTICS ==========")

for column in blocking_columns:

    unique_count = source1[column].nunique()

    empty_count = (
        source1[column] == ""
    ).sum()

    print(
        f"{column:25s} "
        f"unique={unique_count:,} "
        f"empty={empty_count:,}"
    )

Source1 → Source3 candidates: 38,935,086


Generate S1 → S2 candidates

In [ ]:
print("=" * 60)
print("GENERATING S1 → S2 CANDIDATES")
print("=" * 60)

s1_s2_candidates = generate_all_candidates(
    source1,
    source2
)

print(
    f"\nS1 → S2 candidates: "
    f"{len(s1_s2_candidates):,}"
)

True S2 pairs in sample: 176,946


Generate S1 → S3 candidates

In [ ]:
print("=" * 60)
print("GENERATING S1 → S3 CANDIDATES")
print("=" * 60)

s1_s3_candidates = generate_all_candidates(
    source1,
    source3
)

print(
    f"\nS1 → S3 candidates: "
    f"{len(s1_s3_candidates):,}"
)

True S2 pairs          : 176,946
Retained by blocking   : 97,386
Lost by blocking       : 79,560

S1 → S2 blocking recall: 55.04%


S1 → S2 blocking recall

In [ ]:
true_s2 = sample_truth[
    sample_truth["target_source"] == "S2"
].copy()

true_s2_keys = pd.MultiIndex.from_frame(
    true_s2[
        [
            "source1_entity_id",
            "matched_entity_id"
        ]
    ]
)

candidate_s2_keys = pd.MultiIndex.from_frame(
    s1_s2_candidates[
        [
            "entity_id_s1",
            "entity_id_s2"
        ]
    ].rename(
        columns={
            "entity_id_s1": "source1_entity_id",
            "entity_id_s2": "matched_entity_id"
        }
    )
)

retained_s2 = true_s2_keys.isin(
    candidate_s2_keys
)

retained_count_s2 = retained_s2.sum()

true_count_s2 = len(true_s2)

lost_count_s2 = (
    true_count_s2 -
    retained_count_s2
)

recall_s2 = (
    retained_count_s2 / true_count_s2
    if true_count_s2 > 0
    else 0
)

print("========== S1 → S2 ==========")

print(
    f"True S2 pairs          : "
    f"{true_count_s2:,}"
)

print(
    f"Retained by blocking   : "
    f"{retained_count_s2:,}"
)

print(
    f"Lost by blocking       : "
    f"{lost_count_s2:,}"
)

print(
    f"S1 → S2 blocking recall: "
    f"{recall_s2:.2%}"
)

True S3 pairs in sample: 189,518


S1 → S3 blocking recall

In [ ]:
true_s3 = sample_truth[
    sample_truth["target_source"] == "S3"
].copy()

true_s3_keys = pd.MultiIndex.from_frame(
    true_s3[
        [
            "source1_entity_id",
            "matched_entity_id"
        ]
    ]
)

candidate_s3_keys = pd.MultiIndex.from_frame(
    s1_s3_candidates[
        [
            "entity_id_s1",
            "entity_id_s2"
        ]
    ].rename(
        columns={
            "entity_id_s1": "source1_entity_id",
            "entity_id_s2": "matched_entity_id"
        }
    )
)

retained_s3 = true_s3_keys.isin(
    candidate_s3_keys
)

retained_count_s3 = retained_s3.sum()

true_count_s3 = len(true_s3)

lost_count_s3 = (
    true_count_s3 -
    retained_count_s3
)

recall_s3 = (
    retained_count_s3 / true_count_s3
    if true_count_s3 > 0
    else 0
)

print("========== S1 → S3 ==========")

print(
    f"True S3 pairs          : "
    f"{true_count_s3:,}"
)

print(
    f"Retained by blocking   : "
    f"{retained_count_s3:,}"
)

print(
    f"Lost by blocking       : "
    f"{lost_count_s3:,}"
)

print(
    f"S1 → S3 blocking recall: "
    f"{recall_s3:.2%}"
)

True S3 pairs          : 189,518
Retained by blocking   : 107,032
Lost by blocking       : 82,486

S1 → S3 blocking recall: 56.48%


Find lost S2 matches

In [ ]:
retained_s2_set = set(
    zip(
        s1_s2_candidates["entity_id_s1"],
        s1_s2_candidates["entity_id_s2"],
    )
)

lost_s2 = true_s2[
    ~true_s2.apply(
        lambda row: (
            row["source1_entity_id"],
            row["matched_entity_id"]
        ) in retained_s2_set,
        axis=1
    )
].copy()

print("========== LOST S2 MATCHES ==========")

print(
    f"Lost S2 pairs: {len(lost_s2):,}"
)

print(
    lost_s2.head(20).to_string(index=False)
)

Genuinely lost S2 matches: 79,560


,entity_id_s1,entity_id_s2
1,S1-965667,S2-743505751
9,S1-343815751,S2-790675320
10,S1-343815751,S2-479876582
12,S1-656753428,S2-153058913
13,S1-656753428,S2-24659151
15,S1-102811957,S2-478959098
16,S1-102811957,S2-553508714
17,S1-102811957,S2-625774905
21,S1-18727616,S2-755677256
25,S1-318373630,S2-660036492


Find lost S3 matches

In [ ]:
retained_s3_set = set(
    zip(
        s1_s3_candidates["entity_id_s1"],
        s1_s3_candidates["entity_id_s2"],
    )
)

lost_s3 = true_s3[
    ~true_s3.apply(
        lambda row: (
            row["source1_entity_id"],
            row["matched_entity_id"]
        ) in retained_s3_set,
        axis=1
    )
].copy()

print("========== LOST S3 MATCHES ==========")

print(
    f"Lost S3 pairs: {len(lost_s3):,}"
)

print(
    lost_s3.head(20).to_string(index=False)
)

Genuinely lost S3 matches: 82,486


,entity_id_s1,entity_id_s3
2,S1-965667,S3-775321672
3,S1-965667,S3-11291185
4,S1-965667,S3-860443364
14,S1-656753428,S3-679606215
18,S1-102811957,S3-728090388
20,S1-102811957,S3-449308785
22,S1-18727616,S3-187831601
24,S1-18727616,S3-476250621
26,S1-318373630,S3-804600254
28,S1-86989137,S3-312496301


Inspect actual lost S2 records

In [ ]:
lost_s2_sample = lost_s2.head(20)

lost_s2_s1 = source1[
    source1["entity_id"].isin(
        lost_s2_sample["source1_entity_id"]
    )
][
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized",
    ]
].copy()

lost_s2_target = source2[
    source2["entity_id"].isin(
        lost_s2_sample["matched_entity_id"]
    )
][
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized",
    ]
].copy()

print("========== LOST S2 SOURCE1 RECORDS ==========")

print(
    lost_s2_s1.to_string(index=False)
)

print("\n========== LOST S2 TARGET RECORDS ==========")

print(
    lost_s2_target.to_string(index=False)
)


S1: S1-965667
S2: S2-743505751


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
1286323,S1-965667,Maure Williams Colombier Inc,maure williams colombier inc,"85 Wayne Avenue, Ticonderoga, NY",85 wayne avenue ticonderoga ny,US


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
2003754,S2-743505751,Maure Williams Colombier,maure williams colombier,,,US



S1: S1-343815751
S2: S2-790675320


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
547338,S1-343815751,Dahlia Power Reliable Scientific LLC,dahlia power reliable scientific llc,"630 45th Terrace, Kansas City, MO",630 45th terrace kansas city mo,US


...

,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
2117972,S1-18727616,Lumay Boral,lumay boral,"1056 Belden Avenue, Akron, OH",1056 belden avenue akron oh,US


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
223808,S2-755677256,Lumay Boral Inc.,lumay boral inc,"1056-1060 BELDEN AVE, PO BOX 8807, AKRON, OH",1056 1060 belden ave po box 8807 akron oh,US



S1: S1-318373630
S2: S2-660036492


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
603899,S1-318373630,Red Ventures Private Limited,red ventures private limited,"Rajasthan, Jaipur, Banipark, Gokul Apartment, ...",rajasthan jaipur banipark gokul apartment e 3a...,India


,entity_id,business_name,business_name_normalized,business_address,business_address_normalized,country
809231,S2-660036492,रेड वेंचर्स प्राइवेट लिमिटेड,र ड व चर स प र इव ट ल म ट ड,"G-1, BANIPARK, JAIPUR, Rajasthan",g 1 banipark jaipur rajasthan,India


Inspect actual lost S3 records

In [ ]:
lost_s3_sample = lost_s3.head(20)

lost_s3_s1 = source1[
    source1["entity_id"].isin(
        lost_s3_sample["source1_entity_id"]
    )
][
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized",
    ]
].copy()

lost_s3_target = source3[
    source3["entity_id"].isin(
        lost_s3_sample["matched_entity_id"]
    )
][
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "business_name_normalized",
        "business_address_normalized",
    ]
].copy()

print("========== LOST S3 SOURCE1 RECORDS ==========")

print(
    lost_s3_s1.to_string(index=False)
)

print("\n========== LOST S3 TARGET RECORDS ==========")

print(
    lost_s3_target.to_string(index=False)
)

========== S1 → S2 ==========
Possible pairs : 503,461,600,000
Candidates     : 31,784,568
Reduction      : 99.9937%

========== S1 → S3 ==========
Possible pairs : 528,560,300,000
Candidates     : 38,935,086
Reduction      : 99.9926%


Candidate reduction S2

In [ ]:
possible_s2_pairs = (
    len(source1) *
    len(source2)
)

candidate_s2_count = len(
    s1_s2_candidates
)

reduction_s2 = (
    1 -
    candidate_s2_count /
    possible_s2_pairs
)

print("========== S1 → S2 ==========")

print(
    f"Possible pairs : "
    f"{possible_s2_pairs:,}"
)

print(
    f"Candidates     : "
    f"{candidate_s2_count:,}"
)

print(
    f"Reduction      : "
    f"{reduction_s2:.4%}"
)

========== S1 → S2 ==========
Average: 335.44
Median : 39.00
Maximum: 9,049

========== S1 → S3 ==========
Average: 408.37
Median : 38.00
Maximum: 11,602


Candidate reduction S3

In [ ]:
possible_s3_pairs = (
    len(source1) *
    len(source3)
)

candidate_s3_count = len(
    s1_s3_candidates
)

reduction_s3 = (
    1 -
    candidate_s3_count /
    possible_s3_pairs
)

print("========== S1 → S3 ==========")

print(
    f"Possible pairs : "
    f"{possible_s3_pairs:,}"
)

print(
    f"Candidates     : "
    f"{candidate_s3_count:,}"
)

print(
    f"Reduction      : "
    f"{reduction_s3:.4%}"
)

✓ Sample blocking results saved


S2 candidate distribution

In [ ]:
s2_candidate_counts = (
    s1_s2_candidates
    .groupby("entity_id_s1")
    .size()
)

print("========== S1 → S2 ==========")

print(
    f"Average: "
    f"{s2_candidate_counts.mean():.2f}"
)

print(
    f"Median : "
    f"{s2_candidate_counts.median():.2f}"
)

print(
    f"Maximum: "
    f"{s2_candidate_counts.max():,}"
)

S3 candidate distribution

In [ ]:
s3_candidate_counts = (
    s1_s3_candidates
    .groupby("entity_id_s1")
    .size()
)

print("========== S1 → S3 ==========")

print(
    f"Average: "
    f"{s3_candidate_counts.mean():.2f}"
)

print(
    f"Median : "
    f"{s3_candidate_counts.median():.2f}"
)

print(
    f"Maximum: "
    f"{s3_candidate_counts.max():,}"
)

Final summary

In [ ]:
print("\n")
print("=" * 65)
print("FINAL BLOCKING SUMMARY")
print("=" * 65)

print("\nS1 → S2")
print("-" * 30)

print(
    f"True pairs       : {true_count_s2:,}"
)

print(
    f"Candidates       : {candidate_s2_count:,}"
)

print(
    f"Recall           : {recall_s2:.2%}"
)

print(
    f"Candidate reduce : {reduction_s2:.4%}"
)

print("\nS1 → S3")
print("-" * 30)

print(
    f"True pairs       : {true_count_s3:,}"
)

print(
    f"Candidates       : {candidate_s3_count:,}"
)

print(
    f"Recall           : {recall_s3:.2%}"
)

print(
    f"Candidate reduce : {reduction_s3:.4%}"
)

print("\n" + "=" * 65)

Save blocking results

In [ ]:
from src.config import EXPERIMENTS_DIR, OUTPUT_DIR

EXPERIMENTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

blocking_summary = pd.DataFrame([
    {
        "source_pair": "S1-S2",
        "true_pairs": true_count_s2,
        "candidates": candidate_s2_count,
        "recall": recall_s2,
        "candidate_reduction": reduction_s2,
        "average_candidates_per_source1":
            s2_candidate_counts.mean(),
        "median_candidates_per_source1":
            s2_candidate_counts.median(),
        "maximum_candidates_per_source1":
            s2_candidate_counts.max(),
    },
    {
        "source_pair": "S1-S3",
        "true_pairs": true_count_s3,
        "candidates": candidate_s3_count,
        "recall": recall_s3,
        "candidate_reduction": reduction_s3,
        "average_candidates_per_source1":
            s3_candidate_counts.mean(),
        "median_candidates_per_source1":
            s3_candidate_counts.median(),
        "maximum_candidates_per_source1":
            s3_candidate_counts.max(),
    },
])

blocking_summary

Save CSV

In [ ]:
blocking_results_file = (
    EXPERIMENTS_DIR /
    "blocking_results.csv"
)

blocking_summary.to_csv(
    blocking_results_file,
    index=False
)

print(
    f"✓ Blocking results saved to:\n"
    f"{blocking_results_file}"
)

<!-- measured-pipeline-supplement -->
# Production bounded blocking results

Reproducible views of measured CLI artifacts. Historical cells above are preserved; do not rerun the original full-memory blocking experiment on a memory-limited machine. New cells are supplied unexecuted.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.read_csv(ROOT / 'experiments/blocking_results.csv')